# 05 — Error Analysis

## 1. Objective

The goal of this notebook is to understand where the current best model makes mistakes and identify patterns that may help improve model performance.

We focus on:

- False Negatives: actual `Yes`, predicted `No`
- False Positives: actual `No`, predicted `Yes`
- Error patterns across important features
- Possible ideas for model improvement

In [1]:
import sys
sys.path.append("..")

from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from src.data_split import load_and_split_data
from src.preprocessing import build_tree_preprocessor

X_train, X_valid, y_train, y_valid = load_and_split_data()

xgb_model = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_valid)

## 3. Build the Error Table

To analyze model mistakes, we combine the validation features with:

- the actual target value;
- the model prediction.

This allows us to identify which observations are True Positives, True Negatives, False Positives, and False Negatives.

In [2]:
error_df = X_valid.copy()

error_df["actual"] = y_valid
error_df["predicted"] = xgb_pred

display(error_df.head())

,country,year,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type,actual,predicted
69,Kenya,2018,Urban,Yes,1,30,Male,Head of Household,Single/Never Married,Secondary education,Formally employed Private,1,1
13395,Rwanda,2016,Urban,Yes,4,30,Female,Spouse,Married/Living together,Primary education,Remittance Dependent,0,0
18888,Tanzania,2017,Urban,Yes,1,34,Female,Head of Household,Married/Living together,No formal education,Self employed,0,0
4660,Kenya,2018,Rural,No,7,45,Female,Spouse,Married/Living together,Primary education,Informally employed,0,0
13090,Rwanda,2016,Rural,Yes,5,35,Female,Spouse,Married/Living together,Primary education,Farming and Fishing,0,0


## 4. Label Prediction Types

Each validation observation is labeled as:

- True Positive (`TP`)
- True Negative (`TN`)
- False Positive (`FP`)
- False Negative (`FN`)

This makes it easier to isolate and study different types of model errors.

In [3]:
def get_prediction_type(row):
    if row["actual"] == 1 and row["predicted"] == 1:
        return "TP"
    elif row["actual"] == 0 and row["predicted"] == 0:
        return "TN"
    elif row["actual"] == 0 and row["predicted"] == 1:
        return "FP"
    else:
        return "FN"


error_df["prediction_type"] = error_df.apply(
    get_prediction_type,
    axis=1
)

print(error_df["prediction_type"].value_counts())

prediction_type
TN    3960
FN     426
TP     236
FP      83
Name: count, dtype: int64


## 5. Analyze False Negatives

False Negatives are respondents who actually have a bank account (`1`) but were predicted as not having one (`0`).

We analyze these cases to understand whether the model systematically misses particular groups.

In [4]:
false_negatives = error_df[
    error_df["prediction_type"] == "FN"
]

print("Number of False Negatives:", len(false_negatives))

display(false_negatives.head())

Number of False Negatives: 426


,country,year,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type,actual,predicted,prediction_type
21900,Uganda,2018,Urban,Yes,5,31,Female,Head of Household,Divorced/Seperated,Primary education,Self employed,1,0,FN
16125,Tanzania,2017,Urban,Yes,2,80,Female,Spouse,Single/Never Married,No formal education,Self employed,1,0,FN
1104,Kenya,2018,Urban,Yes,2,23,Female,Other non-relatives,Single/Never Married,Tertiary education,Remittance Dependent,1,0,FN
2902,Kenya,2018,Rural,Yes,2,28,Male,Child,Single/Never Married,Secondary education,Informally employed,1,0,FN
8374,Rwanda,2016,Urban,Yes,7,71,Male,Head of Household,Married/Living together,Secondary education,Informally employed,1,0,FN


In [5]:
features_to_check = [
    "country",
    "cellphone_access",
    "education_level",
    "job_type"
]

for col in features_to_check:
    print(f"\n--- {col} ---")
    print(false_negatives[col].value_counts(normalize=True).round(3) * 100)
    


--- country ---
country
Kenya       42.5
Rwanda      35.9
Tanzania    16.7
Uganda       4.9
Name: proportion, dtype: float64

--- cellphone_access ---
cellphone_access
Yes    94.6
No      5.4
Name: proportion, dtype: float64

--- education_level ---
education_level
Primary education                  47.4
Secondary education                33.6
No formal education                 7.7
Tertiary education                  5.6
Vocational/Specialised training     5.4
Other/Dont know/RTA                 0.2
Name: proportion, dtype: float64

--- job_type ---
job_type
Self employed                   27.2
Farming and Fishing             26.5
Informally employed             18.5
Remittance Dependent            10.8
Formally employed Private        6.3
Other Income                     5.4
Government Dependent             3.1
Formally employed Government     0.9
Dont Know/Refuse to answer       0.7
No Income                        0.5
Name: proportion, dtype: float64
